# Wetterdaten importieren und plotten

Dieses Notebook zeigt moeglichst einfach und gut kommentiert, wie du Wetterdaten von der Geosphere-API laedst, in ein `pandas`-`DataFrame` umwandelst, einen sauberen Plot erzeugst und die Ergebnisse exportierst.

Die Schritte sind absichtlich kurz gehalten:

1. Station in den Metadaten finden
2. Wetterdaten fuer diese Station laden
3. JSON in ein DataFrame umwandeln
4. Einen uebersichtlichen Plot erstellen
5. Daten und Plot speichern

## Datenquelle

Wir verwenden dieselbe Quelle wie im alten Notebook:

- Metadaten: `https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-10min/metadata`
- Messdaten: `https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-10min`

Es handelt sich um historische 10-Minuten-Wetterdaten von Geosphere Austria.

In [ ]:
from datetime import datetime, timedelta
from io import BytesIO
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen
import base64
import json

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print("Bibliotheken geladen.")
print("Wir verwenden bewusst nur Standardbibliothek + pandas + matplotlib.")

## 1. Einfache Konfiguration

Hier stellst du die Station und den Zeitraum ein. Fuer den Anfang ist `Lienz` eine gute Beispielstation, weil sie auch im alten Notebook vorkam.

In [ ]:
STATION_NAME = "Lienz"
DAYS_BACK = 2
PARAMETERS = ["TL", "RF", "P", "FFAM", "FFX", "RR"]

# Die Ausgabepfade sammeln wir zentral, damit spaeter klar ist, wo alles landet.
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
PLOTS_DIR = NOTEBOOK_DIR / "plots"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

EXCEL_PATH = DATA_DIR / "weather_data_lienz.xlsx"
PDF_PATH = PLOTS_DIR / "weather_plot_lienz.pdf"
JPG_PATH = PLOTS_DIR / "weather_plot_lienz.jpg"
HTML_PATH = PLOTS_DIR / "weather_plot_lienz.html"

print(f"Station: {STATION_NAME}")
print(f"Zeitraum: letzte {DAYS_BACK} Tage")
print(f"Parameter: {', '.join(PARAMETERS)}")
print(f"Excel-Datei: {EXCEL_PATH.resolve()}")
print(f"Plot-Ordner: {PLOTS_DIR.resolve()}")

## 2. Kleine Hilfsfunktion fuer API-Aufrufe

Die Funktion unten laedt JSON von einer URL. Sie ist absichtlich sehr klein gehalten, damit man den Ablauf gut nachvollziehen kann.

In [ ]:
def load_json(url: str, params: dict | None = None) -> dict:
    """Lade JSON von einer URL und gib es als Python-Dictionary zurueck."""
    full_url = url
    if params:
        full_url = f"{url}?{urlencode(params)}"

    with urlopen(full_url, timeout=120) as response:
        return json.load(response)

## 3. Station in den Metadaten suchen

Zuerst laden wir die Stationsliste. Danach filtern wir nach dem gewuenschten Stationsnamen.

In [ ]:
META_URL = "https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-10min/metadata"
meta = load_json(META_URL)

print("Metadaten wurden geladen.")
print("Die wichtigsten Bereiche im JSON sind:")
display(list(meta.keys()))

In [ ]:
stations_df = pd.DataFrame(meta["stations"])
print("Die Stationsliste wurde in ein DataFrame umgewandelt.")
display(stations_df[["id", "name", "state", "lat", "lon", "altitude", "is_active"]].head())

In [ ]:
station_matches = stations_df[stations_df["name"].str.contains(STATION_NAME, case=False, na=False)].copy()

print(f"Gefundene Stationen fuer '{STATION_NAME}':")
display(station_matches[["id", "name", "state", "altitude", "is_active"]])

if station_matches.empty:
    raise ValueError(f"Keine Station fuer '{STATION_NAME}' gefunden.")

In [ ]:
# Fuer das simple Beispiel nehmen wir den ersten Treffer.
station_row = station_matches.iloc[0]
station_id = int(station_row["id"])

print(f"Verwendete Station: {station_row['name']} (ID: {station_id})")
print(f"Bundesland: {station_row['state']}")
print(f"Hoehe: {station_row['altitude']} m")

## 4. Wetterdaten fuer die Station laden

Jetzt fragen wir die eigentlichen Messwerte ab. Wir holen Temperatur, Feuchte, Druck, Wind und Niederschlag fuer die letzten zwei Tage.

In [ ]:
DATA_URL = "https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-10min"

end = datetime.now()
start = end - timedelta(days=DAYS_BACK)

query = {
    "station_ids": station_id,
    "parameters": ",".join(PARAMETERS),
    "start": start.strftime("%Y-%m-%dT%H:%M"),
    "end": end.strftime("%Y-%m-%dT%H:%M"),
}

weather_json = load_json(DATA_URL, query)

print("Wetterdaten wurden geladen.")
print("Im JSON sehen wir unter anderem Zeitstempel und Features mit den Parameterdaten.")
display(list(weather_json.keys()))

## 5. JSON in ein DataFrame umwandeln

Die API liefert:

- eine Liste von Zeitstempeln
- dazu pro Parameter eine Datenreihe

Wir bauen daraus ein normales Tabellenformat, das man leicht plotten und exportieren kann.

In [ ]:
feature = weather_json["features"][0]
parameter_dict = feature["properties"]["parameters"]

time_index = pd.to_datetime(weather_json["timestamps"], utc=True).tz_convert("Europe/Vienna")
df = pd.DataFrame(index=time_index)

for parameter_code, parameter_info in parameter_dict.items():
    column_name = f"{parameter_info['name']} ({parameter_info['unit']})"
    df[column_name] = parameter_info["data"]

df.index.name = "timestamp"

print("Das DataFrame ist jetzt die zentrale Arbeitsgrundlage.")
print("Jede Zeile entspricht einem 10-Minuten-Zeitpunkt.")
display(df.head())

In [ ]:
print("Die Spaltennamen enthalten den Messnamen und die Einheit.")
display(pd.Series(df.columns, name="Spalten"))

print("describe(): schnelle Uebersicht ueber die numerischen Werte.")
display(df.describe())

## 6. Einen einfachen und schoenen Plot bauen

Wir erstellen drei uebereinanderliegende Teilplots:

- Temperatur und relative Feuchte
- Wind und Spitzenboeen
- Luftdruck und Niederschlag

Jeder Teilplot hat links und rechts eine eigene y-Achse. So lassen sich unterschiedliche Groessen trotzdem gut zusammen anschauen.

In [ ]:
def plot_weather(df: pd.DataFrame):
    """Erzeuge einen gut lesbaren Wetterplot mit drei Teilbereichen."""
    pairs = [
        ("Lufttemperatur 2m (°C)", "Relative Feuchte (%)"),
        ("Windgeschwindigkeit 10m, arithmetischer Mittelwert (m/s)", "Maximale Windgeschwindigkeit (Spitzenböe) (m/s)"),
        ("Luftdruck (hPa)", "Niederschlagssumme (mm)"),
    ]

    colors = [
        ("tab:red", "tab:blue"),
        ("tab:green", "tab:orange"),
        ("tab:purple", "tab:gray"),
    ]

    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(15, 10), sharex=True, constrained_layout=True)
    fig.suptitle(f"Wetterdaten fuer {station_row['name']} - letzte {DAYS_BACK} Tage", fontsize=16)

    for ax, (left_col, right_col), (left_color, right_color) in zip(axes, pairs, colors):
        # Linke Achse fuer die erste Messgroesse
        ax.plot(df.index, df[left_col], color=left_color, linewidth=1.8, label=left_col)
        ax.set_ylabel(left_col, color=left_color)
        ax.tick_params(axis="y", labelcolor=left_color)
        ax.grid(True, alpha=0.3)

        # Rechte Achse fuer die zweite Messgroesse
        ax_right = ax.twinx()
        ax_right.plot(df.index, df[right_col], color=right_color, linewidth=1.5, linestyle="--", label=right_col)
        ax_right.set_ylabel(right_col, color=right_color)
        ax_right.tick_params(axis="y", labelcolor=right_color)

        # Beide Linien in einer gemeinsamen Legende zusammenfassen
        lines = ax.get_lines() + ax_right.get_lines()
        labels = [line.get_label() for line in lines]
        ax.legend(lines, labels, loc="upper left")

    axes[-1].set_xlabel("Zeit")
    return fig

In [ ]:
print("Jetzt erzeugen wir den Plot und zeigen ihn direkt im Notebook an.")
fig = plot_weather(df)
plt.show()

## 7. Daten und Plots exportieren

Wir speichern:

- die Daten als Excel-Datei (`.xlsx`)
- den Plot als `.pdf`
- den Plot als `.jpg`
- den Plot als einfache `.html`-Datei

Die HTML-Datei ist hier absichtlich simpel: Sie enthaelt den Plot als eingebettetes Bild plus eine kurze Zusammenfassung. So bleibt alles ohne Zusatzbibliotheken nachvollziehbar.

In [ ]:
print("1. Wir speichern die Wetterdaten als Excel-Datei.")
print("   Excel kann keine Zeitzonen in Datumswerten speichern, deshalb machen wir Zeitstempel vorher tz-unaware.")

# Wir arbeiten auf einer Kopie, damit der DataFrame fuer Plot und Analyse unveraendert bleibt.
df_excel = df.copy()

# Falls der Zeitstempel im Index eine Zeitzone hat, entfernen wir nur die TZ-Info.
# Die Uhrzeit selbst bleibt dabei erhalten, Excel bekommt also normale Datumswerte ohne Zeitzone.
if isinstance(df_excel.index, pd.DatetimeIndex) and df_excel.index.tz is not None:
    df_excel.index = df_excel.index.tz_localize(None)

# Dasselbe gilt fuer eventuelle datetime-Spalten mit Zeitzone.
for col in df_excel.select_dtypes(include=["datetimetz"]).columns:
    df_excel[col] = df_excel[col].dt.tz_localize(None)

df_excel.to_excel(EXCEL_PATH, engine="xlsxwriter")

print("2. Wir speichern denselben Plot als PDF und JPG.")
fig.savefig(PDF_PATH, bbox_inches="tight")
fig.savefig(JPG_PATH, dpi=200, bbox_inches="tight")

print("3. Fuer HTML rendern wir den Plot in einen Speicherpuffer und betten das Bild dann direkt ein.")
buffer = BytesIO()
fig.savefig(buffer, format="jpg", dpi=160, bbox_inches="tight")
buffer.seek(0)
image_base64 = base64.b64encode(buffer.read()).decode("ascii")

summary_html = f"""
<!DOCTYPE html>
<html lang='de'>
<head>
  <meta charset='utf-8'>
  <title>Wetterplot {station_row['name']}</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 2rem; line-height: 1.5; }}
    h1 {{ margin-bottom: 0.2rem; }}
    .meta {{ color: #444; margin-bottom: 1.5rem; }}
    img {{ max-width: 100%; height: auto; border: 1px solid #ddd; }}
    table {{ border-collapse: collapse; margin-top: 1rem; }}
    th, td {{ border: 1px solid #ddd; padding: 0.4rem 0.6rem; text-align: left; }}
  </style>
</head>
<body>
  <h1>Wetterdaten fuer {station_row['name']}</h1>
  <p class='meta'>Zeitraum: {start:%Y-%m-%d %H:%M} bis {end:%Y-%m-%d %H:%M}</p>
  <p>Dieser HTML-Export enthaelt denselben Plot wie die PDF- und JPG-Datei, eingebettet als Bild.</p>
  <img src='data:image/jpeg;base64,{image_base64}' alt='Wetterplot'>
  <h2>Erste Datenzeilen</h2>
  {df.head().to_html()}
</body>
</html>
"""

HTML_PATH.write_text(summary_html, encoding="utf-8")

print("Export fertig.")
print(f"Excel: {EXCEL_PATH.resolve()}")
print(f"PDF:   {PDF_PATH.resolve()}")
print(f"JPG:   {JPG_PATH.resolve()}")
print(f"HTML:  {HTML_PATH.resolve()}")

## 8. Was du am einfachsten anpassen kannst

- `STATION_NAME` aendern, um eine andere Station zu laden
- `DAYS_BACK` vergroessern oder verkleinern
- `PARAMETERS` anpassen, falls du andere Messgroessen brauchst

Wenn du moechtest, kann man das Notebook spaeter noch erweitern. Fuer den Einstieg ist diese Version aber absichtlich kompakt und direkt nachvollziehbar.